In [3]:
# Clean install with compatible versions
import sys

# Install pandas without pyarrow first
!{sys.executable} -m pip install pandas numpy plotly

# Install fastparquet as the parquet engine
!{sys.executable} -m pip install fastparquet torch torchvision torchaudio

# Import libraries
import pandas as pd
import numpy as np
import pickle
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("All packages imported successfully with fastparquet engine!")

# Set fastparquet as default engine to avoid conflicts
pd.options.io.parquet.engine = 'fastparquet'


Python(69209) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Python(69210) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch
All packages imported successfully with fastparquet engine!


## LOAD INPUTS + WAVEFORM NUMERICS

In [4]:

mv_filtered_10min = pd.read_parquet('mv_filtered_10min.parquet')
mv_filtered_10min.head(100)

,subject_id,hadm_id,item_id,input_name,input_class,start_time,end_time,rate,rate_uom,rate/weight,...,trigger,trigger_reason,action_cluster_id,action_cluster_size,action_cluster_rank,first,last,absolute_timestamp,wf_time_delta_s,rate/weight_normalized
0,5171,125124,221906,01-Drips,Continuous Med,2171-10-16 08:05:00+00:00,2171-10-16 08:17:00+00:00,0.100081,mcg/kg/min,0.100081,...,False,,NaN,NaN,NaN,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 08:05:01+00:00,1.0,-0.342774
1,5171,125124,221906,01-Drips,Continuous Med,2171-10-16 08:17:00+00:00,2171-10-16 08:24:00+00:00,0.080050,mcg/kg/min,0.080050,...,False,,NaN,NaN,NaN,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 08:17:01+00:00,1.0,-0.454216
2,5171,125124,221906,01-Drips,Continuous Med,2171-10-16 08:24:00+00:00,2171-10-16 08:39:00+00:00,0.060038,mcg/kg/min,0.060038,...,False,,NaN,NaN,NaN,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 08:24:01+00:00,1.0,-0.565555
3,5171,125124,221906,01-Drips,Continuous Med,2171-10-16 08:39:00+00:00,2171-10-16 08:56:00+00:00,0.040033,mcg/kg/min,0.040033,...,True,decrease,9.0,2.0,1.0,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 08:39:01+00:00,1.0,-0.676852
4,5171,125124,221906,01-Drips,Continuous Med,2171-10-16 08:56:00+00:00,2171-10-16 09:35:00+00:00,0.080065,mcg/kg/min,0.080065,...,True,increase,9.0,2.0,2.0,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 08:56:01+00:00,1.0,-0.454132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,5348,161526,225158,01-Drips,Continuous Med,2181-07-03 01:31:00+00:00,2181-07-03 02:03:00+00:00,7.825194,mL/hour,0.075026,...,False,increase,NaN,NaN,NaN,2181-07-02 14:20:16+00:00,2181-07-04 18:04:06+00:00,2181-07-03 01:30:56+00:00,-4.0,-0.282489
96,5348,161526,225158,01-Drips,Continuous Med,2181-07-03 02:03:00+00:00,2181-07-03 02:25:00+00:00,3.912596,mL/hour,0.037513,...,False,decrease,NaN,NaN,NaN,2181-07-02 14:20:16+00:00,2181-07-04 18:04:06+00:00,2181-07-03 02:02:56+00:00,-4.0,-0.331636
97,5348,161526,221906,01-Drips,Continuous Med,2181-07-03 02:03:00+00:00,2181-07-03 02:25:00+00:00,0.020007,mcg/kg/min,0.020007,...,True,decrease,6.0,1.0,1.0,2181-07-02 14:20:16+00:00,2181-07-04 18:04:06+00:00,2181-07-03 02:02:56+00:00,-4.0,-0.788265
98,5348,161526,225158,01-Drips,Continuous Med,2181-07-03 02:17:00+00:00,2181-07-03 03:24:00+00:00,5.001128,mL/hour,0.047949,...,False,,NaN,NaN,NaN,2181-07-02 14:20:16+00:00,2181-07-04 18:04:06+00:00,2181-07-03 02:16:56+00:00,-4.0,-0.317963


In [34]:
mv_filtered_10min['item_label'].unique()
mv_filtered_10min['rate_weight_norm']

0        0.100081
1        0.080050
2        0.060038
3        0.040033
4        0.080065
           ...   
32405    3.511236
32406    1.404494
32407    4.915730
32408    0.250522
32409    0.125261
Name: rate_weight_norm, Length: 32410, dtype: float64

In [5]:
# Calculate min and max for each group instead of mean and std
stats = mv_filtered_10min.groupby('item_label')['rate/weight'].agg(['min', 'max']).dropna()

# Apply min-max normalization for each group
def normalize_by_group(row):
    item = row['item_label']
    value = row['rate/weight']
    
    if pd.isna(value) or item not in stats.index:
        return np.nan
    
    min_val = stats.loc[item, 'min']
    max_val = stats.loc[item, 'max']
    
    # Avoid division by zero (when min equals max)
    if max_val == min_val:
        return 0

    return (value - min_val) / (max_val - min_val)

# Create the normalized column
mv_filtered_10min['rate/weight_normalized'] = mv_filtered_10min.apply(normalize_by_group, axis=1)
mv_filtered_10min['rate/weight_normalized']

0        0.022994
1        0.018150
2        0.013310
3        0.008472
4        0.018153
           ...   
32405    0.065611
32406    0.051154
32407    0.180123
32408    0.222176
32409    0.095991
Name: rate/weight_normalized, Length: 32410, dtype: float64

In [6]:
mv_filtered_10min.to_parquet('mv_filtered_10min.parquet')

In [25]:

df_clean = pd.read_parquet("combined_waveforms.cleaned.parquet")
df_clean.head()
df_clean.columns

Index(['hadm_id', 'record_name', 'absolute_timestamp', 'ABP MEAN', 'NBP MEAN',
       'CVP', 'HR', 'RESP', 'record_start_time', 'record_end_time',
       'icu_admission_time', 'time_seconds'],
      dtype='object')

In [26]:
abp_valid = df_clean['ABP MEAN'].dropna()
cvp_valid = df_clean['CVP'].dropna()

print(f"ABP Mean - Mean: {abp_valid.mean():.3f}, Std: {abp_valid.std():.3f} (n={len(abp_valid)})")
print(f"CVP - Mean: {cvp_valid.mean():.3f}, Std: {cvp_valid.std():.3f} (n={len(cvp_valid)})")

ABP Mean - Mean: 78.937, Std: 23.009 (n=43369731)
CVP - Mean: 8.505, Std: 7.948 (n=64774057)


Train vs Val vs Test


Split by Hadm-Id + balance the number of triggers 

hadm_trigger_counts = (
    mv_filtered_10min[mv_filtered_10min["trigger"] == True]
    .groupby("hadm_id")
    .size()
    .reset_index(name="n_triggers")
    .sort_values("n_triggers", ascending=False)
)
hadm_trigger_counts


---

Build a dataloader with the following:

for every t0 = action cluster id 


- 'context' information:
    waveform for the 5 vars: ABP Mean, CVP, HR, RESP for the 60 mins before divide into 10 min chunks (x6) and average within each 10 mins: 

    + MASK 

- 'Initial conditions' from wavefroms if exist, if not exist 
    the ones NOT from waveform data we get from encoder OR for now we just randomly sample within the ranges

    + MASK (for future encoder)

- 'Treatments': 
    treatment type
    dose
    time (expoential decay) from start of t0 (be a learned variable) 

    + MASK 

- 'Response':
    waveform ABP Mean +CVP for the following 60 mins

    + MASK: element wise (time) + CVP vs ABP: 


---
BUILD CLUSTER DATASET

In [27]:
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset

class ClusterDataset(Dataset):
    def __init__(self, manifest_csv, samples_dir):
        self.man = pd.read_csv(manifest_csv)
        # ✅ keep only rows with status == "ok"
        if "status" in self.man.columns:
            self.man = self.man[self.man["status"] == "ok"].copy().reset_index(drop=True)

        self.dir = Path(samples_dir)

    def __len__(self):
        return len(self.man)

    def __getitem__(self, idx):
        row = self.man.iloc[idx]
        npz = np.load(self.dir / f"{row['hadm_id']}_{row['action_cluster_id']}.npz")
        return {
            "initial": npz["initial"],
            "initial_mask": npz["initial_mask"],
            "prev_avg": npz["prev_avg"],
            "prev_avg_mask": npz["prev_avg_mask"],
            "response": npz["response"],
            "response_mask": npz["response_mask"],
            "treatments": npz["treatments"],
            "meta": dict(row),
        }

ModuleNotFoundError: No module named 'torch'

## CLEAN WAVEFORM + SMOOTH 

In [29]:
import numpy as np
import pandas as pd

def _smooth_1d_nanaware(arr: np.ndarray, neighbors: int,
                        keep_nan_center: bool = True,
                        min_valid: int = 1) -> np.ndarray:
    if neighbors <= 0 or arr.size == 0:
        return arr.astype(float, copy=True)
    win = 2*neighbors + 1
    s = pd.Series(arr, dtype="float64")
    sm = s.rolling(win, center=True, min_periods=min_valid).mean()
    if keep_nan_center:
        sm[s.isna()] = np.nan
    return sm.to_numpy()

def _clip_inplace(g: pd.DataFrame):
    if "ABP MEAN" in g:
        g["ABP MEAN"] = pd.to_numeric(g["ABP MEAN"], errors="coerce").clip(lower=40, upper=180)
    if "CVP" in g:
        g["CVP"] = pd.to_numeric(g["CVP"], errors="coerce").clip(lower=0, upper=40)
    return g

def _zero_center_cols(g: pd.DataFrame, cols, suffix="_zc"):
    for c in cols:
        if c in g:
            mu = g[c].mean(skipna=True)
            g[f"{c}{suffix}"] = g[c] - mu
    return g

def _zscore_cols(g: pd.DataFrame, cols, suffix="_zn"):
    for c in cols:
        if c in g:
            mu = g[c].mean(skipna=True)
            sd = g[c].std(skipna=True)
            g[f"{c}{suffix}"] = (g[c] - mu) / sd if (pd.notna(sd) and sd > 0) else np.nan
    return g

def _smooth_cols_multi(g: pd.DataFrame, cols, neighbors, source_suffixes, out_suffix):
    """
    Smooth multiple source variants (e.g., raw/zc/zn) in one go.
    For each c in cols and each src in source_suffixes, create c{src}{out_suffix}.
    """
    for c in cols:
        for src in source_suffixes:
            base = f"{c}{src}" if src else c
            if base in g:
                g[f"{base}{out_suffix}"] = _smooth_1d_nanaware(
                    pd.to_numeric(g[base], errors="coerce").to_numpy(),
                    neighbors=neighbors, keep_nan_center=True, min_valid=1
                )
    return g

# goes thru wf database, choose z_score or z_center. Keep signal, abs timestamp, zero_center false, z-score false. 
def run_waveform_pipeline(
    df: pd.DataFrame,
    signals=("ABP MEAN","CVP","HR","RESP"),
    time_col="absolute_timestamp",
    group_cols=("hadm_id","record_name"),
    *,
    do_zero_center: bool = True,
    do_zscore: bool = True,
    smooth_neighbors: int = 120,
    smooth_variants=("zc","zn"),   # <- choose any of {"raw","zc","zn"}; e.g. ("zc","zn")
    out_suffix: str = "_ma120",
    flush_every_rows: int = 2_000_000,
):
    """
    Memory-friendly generator: per (hadm_id, record_name) it clips ABP/CVP,
    optionally creates _zc and/or _zn, then smooths any of the requested variants
    (e.g., zc and zn), yielding chunks so you can concat or write to disk.
    """
    need_time = time_col in df.columns
    out_frames, acc_rows = [], 0
    sort_keys = list(group_cols) + ([time_col] if need_time else [])
    df_sorted = df.sort_values(sort_keys)

    # map variant tokens to suffixes
    var2suf = {"raw": "", "zc": "_zc", "zn": "_zn"}
    src_suffixes = [var2suf[v] for v in smooth_variants]

    for _, g in df_sorted.groupby(list(group_cols), sort=False, dropna=False):
        # ensure numeric for signals
        for c in signals:
            if c in g:
                g[c] = pd.to_numeric(g[c], errors="coerce")

        # 1) clip only ABP MEAN and CVP
        g = _clip_inplace(g)

        # 2) normalization variants (per group)
        if do_zero_center:
            g = _zero_center_cols(g, signals, suffix="_zc")
        if do_zscore:
            g = _zscore_cols(g, signals, suffix="_zn")

        # 3) smoothing for any requested variants
        if smooth_neighbors and smooth_neighbors > 0 and src_suffixes:
            g = _smooth_cols_multi(g, signals, neighbors=smooth_neighbors,
                                   source_suffixes=src_suffixes, out_suffix=out_suffix)

        out_frames.append(g)
        acc_rows += len(g)
        if acc_rows >= flush_every_rows:
            yield pd.concat(out_frames, ignore_index=True)
            out_frames.clear()
            acc_rows = 0

    if out_frames:
        yield pd.concat(out_frames, ignore_index=True)

In [30]:
chunks = []
for chunk in run_waveform_pipeline(
        df_clean,
        signals=("ABP MEAN","CVP","HR","RESP"),
        do_zero_center=True,
        do_zscore=True,
        smooth_neighbors=120,
        smooth_variants=("zc","zn"),  # <— smooth both zero-centered and z-scored
        out_suffix="_ma120",
        flush_every_rows=1_000_000):
    chunks.append(chunk)

df_final = pd.concat(chunks, ignore_index=True)

In [10]:
df_final = pd.read_parquet("combined_waveforms_cleaned_smooth.parquet")
abp_valid = df_final['ABP MEAN'].dropna()
cvp_valid = df_final['CVP'].dropna()

print(f"ABP Mean - Mean: {abp_valid.mean():.3f}, Std: {abp_valid.std():.3f} (n={len(abp_valid)})")
print(f"CVP - Mean: {cvp_valid.mean():.3f}, Std: {cvp_valid.std():.3f} (n={len(cvp_valid)})")


columns_to_normalize = ['ABP MEAN', 'CVP', 'HR', 'RESP']

# Create z-normalized columns
for col in columns_to_normalize:
    if col in df_final.columns:
        # Calculate mean and std after dropping NaN values
        col_mean = df_final[col].dropna().mean()
        col_std = df_final[col].dropna().std()
        
        # Create z-normalized column
        # This will automatically preserve NaN where original values are NaN
        df_final[f'{col}_z'] = (df_final[col] - col_mean) / col_std
    else:
        print(f"Warning: Column '{col}' not found in DataFrame")

# Display the new columns to verify
print("New z-normalized columns created:")
z_columns = [col for col in df_final.columns if col.endswith('_z')]
print(z_columns)

# Show a sample of the data
print("\nSample data (first 5 rows):")
cols_to_show = columns_to_normalize + [col + '_z' for col in columns_to_normalize if col in df_final.columns]
print(df_final[cols_to_show].head())

df_final.to_parquet("combined_waveforms_cleaned_smooth.parquet")


ABP Mean - Mean: 78.812, Std: 18.679 (n=43369731)
CVP - Mean: 8.505, Std: 7.948 (n=64774057)
New z-normalized columns created:
['ABP MEAN_z', 'CVP_z', 'HR_z', 'RESP_z']

Sample data (first 5 rows):
   ABP MEAN    CVP    HR  RESP  ABP MEAN_z     CVP_z      HR_z    RESP_z
0       NaN   9.60  91.0  12.0         NaN  0.137797  0.264305 -1.414881
1       NaN   9.75  91.0  12.0         NaN  0.156670  0.264305 -1.414881
2       NaN   9.90  91.0  12.0         NaN  0.175543  0.264305 -1.414881
3       NaN  10.05  91.0  12.0         NaN  0.194416  0.264305 -1.414881
4       NaN  10.20  91.0  12.0         NaN  0.213289  0.264305 -1.414881


In [32]:
df_final.head()

,hadm_id,record_name,absolute_timestamp,ABP MEAN,NBP MEAN,CVP,HR,RESP,record_start_time,record_end_time,icu_admission_time,time_seconds
0,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:00.056,NaN,NaN,9.60,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,0
1,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:10.056,NaN,NaN,9.75,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,10
2,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:20.056,NaN,NaN,9.90,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,20
3,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:30.056,NaN,NaN,10.05,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,30
4,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:40.056,NaN,NaN,10.20,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,40


## PLOT INPUTS + WAVEFORMS FOR ANY HADM_ID IN MV

In [ ]:
def plot_waveforms_with_mv_inputs(
    combined_waveform_df: pd.DataFrame,
    input_mv_triggers: pd.DataFrame,
    hadm_id=None,
    start=None,
    end=None,
    resample=None,
    signals=("ABP MEAN","NBP MEAN","CVP","HR","RESP"),
    line_width=1.3,
    trigger_only=True,
    time_col="start_time",
    cluster_col="action_cluster",
    signal_variant="raw",              # 'raw' | 'zc' | 'zc_sm' | 'custom'
    custom_suffix=None,                # e.g. "_ma15", "_sm2", "_g"
):
    wf = combined_waveform_df.copy()
    mv = input_mv_triggers.copy()

    # --- suffix selection ---
    suffix_map = {"raw": "", "zc": "_zc", "zc_sm": "_zc_sm"}
    if signal_variant == "custom":
        if not custom_suffix:
            raise ValueError("When signal_variant='custom', provide custom_suffix (e.g., '_ma15').")
        wanted_suffix = custom_suffix
    else:
        if signal_variant not in suffix_map:
            raise ValueError("signal_variant must be one of {'raw','zc','zc_sm','custom'}")
        wanted_suffix = suffix_map[signal_variant]

    # resolve columns to plot
    plot_cols, titles = [], []
    for base in signals:
        candidate = base + wanted_suffix if wanted_suffix else base
        if candidate in wf.columns:
            plot_cols.append(candidate)
            tag = (signal_variant if signal_variant != "raw" else None)
            if signal_variant == "custom": tag = custom_suffix
            titles.append(base if tag is None else f"{base} ({tag})")
        elif base in wf.columns:
            plot_cols.append(base)   # fallback
            titles.append(f"{base} (fallback)")
        else:
            plot_cols.append(None)
            titles.append(f"{base} (missing)")

    # --- timestamps + filtering unchanged ---
    wf["absolute_timestamp"] = pd.to_datetime(wf["absolute_timestamp"], errors="coerce", utc=True)
    mv[time_col]             = pd.to_datetime(mv[time_col], errors="coerce", utc=True)
    if hadm_id is not None:
        if "hadm_id" in wf.columns: wf = wf[wf["hadm_id"] == hadm_id]
        if "hadm_id" in mv.columns: mv = mv[mv["hadm_id"] == hadm_id]
    if trigger_only and "trigger" in mv.columns:
        mv = mv[mv["trigger"] == True]

    # auto window
    if start is None or end is None:
        mins = [t for t in [wf["absolute_timestamp"].min(), mv[time_col].min()] if pd.notna(t)]
        maxs = [t for t in [wf["absolute_timestamp"].max(), mv[time_col].max()] if pd.notna(t)]
        if not mins or not maxs:
            raise ValueError("No timestamps found after filtering; check hadm_id or inputs.")
        start = min(mins) if start is None else pd.to_datetime(start, utc=True)
        end   = max(maxs) if end   is None else pd.to_datetime(end,   utc=True)
    else:
        start = pd.to_datetime(start, utc=True); end = pd.to_datetime(end, utc=True)

    wf = wf[(wf["absolute_timestamp"] >= start) & (wf["absolute_timestamp"] <= end)]
    mv = mv[(mv[time_col] >= start) & (mv[time_col] <= end)]
    wf = wf.sort_values("absolute_timestamp"); mv = mv.sort_values(time_col)

    if resample:
        keep_cols = ["absolute_timestamp", *[c for c in plot_cols if c]]
        wf = (wf[keep_cols].set_index("absolute_timestamp").resample(resample).mean().reset_index())

    # --- plotting unchanged below ---
    nrows = len(signals)
    fig = make_subplots(rows=nrows, cols=1, shared_xaxes=True, vertical_spacing=0.02, subplot_titles=titles)
    clusters = mv[cluster_col].dropna().astype(str).unique() if cluster_col in mv.columns else []
    palette = px.colors.qualitative.Set2 + px.colors.qualitative.Set1 + px.colors.qualitative.Plotly
    color_map = {lab: palette[i % len(palette)] for i, lab in enumerate(sorted(clusters))}

    y_limits = {}
    for r, (base, col) in enumerate(zip(signals, plot_cols), start=1):
        if not col or col not in wf.columns:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", showlegend=False), row=r, col=1)
            y_limits[base] = (0, 1); continue
        y = pd.to_numeric(wf[col], errors="coerce")
        fig.add_trace(go.Scattergl(x=wf["absolute_timestamp"], y=y, mode="lines", name=col,
                                   line=dict(width=line_width), showlegend=False), row=r, col=1)
        finite = np.isfinite(y.to_numpy())
        if finite.any():
            ymin, ymax = np.nanmin(y), np.nanmax(y)
            y_limits[base] = (ymin - 0.05*(ymax-ymin) if ymin!=ymax else float(ymin)-1,
                              ymax + 0.05*(ymax-ymin) if ymin!=ymax else float(ymax)+1)
        else:
            y_limits[base] = (0, 1)

    for r, base in enumerate(signals, start=1):
        ymin, ymax = y_limits[base]
        for lab in sorted(clusters):
            times = mv.loc[mv[cluster_col].astype(str) == lab, time_col]
            if times.empty: continue
            first = True
            for t in times:
                fig.add_trace(go.Scatter(x=[t, t], y=[ymin, ymax], mode="lines",
                                         line=dict(color=color_map[lab], width=1.3, dash="dot"),
                                         name=str(lab), legendgroup=str(lab),
                                         showlegend=(r == 1 and first),
                                         hoverinfo="text", text=[f"{lab}<br>{t}", f"{lab}<br>{t}"]), row=r, col=1)
                first = False

    fig.update_layout(height=220*nrows, hovermode="x unified",
                      margin=dict(t=40,b=40,l=50,r=10), legend_title_text="MV action clusters")
    for r, base in enumerate(signals, start=1):
        fig.update_yaxes(range=list(y_limits[base]), row=r, col=1, title_text=base)
    fig.update_xaxes(title_text="Time", range=[start, end])
    return fig

In [ ]:
fig = plot_waveforms_with_mv_inputs(df_clean, mv_filtered_10min,
    hadm_id=192494, cluster_col="action_cluster_id",
    signal_variant="raw")
fig.show()


In [ ]:
# plot the smoothed columns by asking the plotter for a custom suffix:
fig = plot_waveforms_with_mv_inputs(
    df_clean,
    mv_filtered_10min,
    hadm_id=101662,
    cluster_col="action_cluster_id",
    signal_variant="raw",
    custom_suffix="_ma120"               # uses e.g. 'ABP MEAN_ma15'
)
fig.show()